# Denoising Visualization: mid_trigger_ratio Sweep

扫 `mid_trigger_ratio` 0.1 ~ 0.9，同样 50 个 GSM8K（seed=42），
每个 ratio 生成一个 HTML 可视化文件。

| ratio | 含义 |
|-------|------|
| 0.1 | 只需解码 10% 就触发 mid-block expand |
| 0.9 | 需解码 90% 才触发 expand |

**输出**: `viz_ratio_0.10.html` ~ `viz_ratio_0.90.html`（各含 50 个 sample 的逐步 denoising）

## 1. 环境设置

In [ ]:
import os, sys, gc
import torch
import numpy as np

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
os.chdir(os.path.join(NOTEBOOK_DIR, 'llada'))
print(f'Working dir: {os.getcwd()}')

if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

torch.cuda.empty_cache(); gc.collect()
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB')

## 2. 模型 & 数据加载

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from model.modeling_llada import LLaDAModelLM
from datasets import load_dataset

MODEL_PATH = 'GSAI-ML/LLaDA-8B-Instruct'
MASK_ID = 126336

config = AutoConfig.from_pretrained(MODEL_PATH)
config.flash_attention = True
model = LLaDAModelLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16, config=config,
).eval().to('cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

gsm8k = load_dataset('gsm8k', 'main', split='test')
print(f'Model loaded. GSM8K test: {len(gsm8k)} samples')

## 3. 构建 prompt（5-shot，seed=42 锁定前 50 题）

In [ ]:
import re

FEW_SHOT_EXAMPLES = """Question: Jen and Tyler are gymnasts practicing flips. Jen is practicing the triple-flip while Tyler is practicing the double-flip. Jen did sixteen triple-flips during practice. Tyler flipped in the air half the number of times Jen did. How many double-flips did Tyler do?
Answer: Jen did 16 triple-flips, so she did 16 * 3 = <<16*3=48>>48 flips.
Tyler did half the number of flips, so he did 48 / 2 = <<48/2=24>>24 flips.
A double flip has two flips, so Tyler did 24 / 2 = <<24/2=12>>12 double-flips.
#### 12

Question: Four people in a law firm are planning a party. Mary will buy a platter of pasta for $20 and a loaf of bread for $2. Elle and Andrea will split the cost for buying 4 cans of soda which cost $1.50 each, and chicken wings for $10. Joe will buy a cake that costs $5. How much more will Mary spend than the rest of the firm put together?
Answer: Mary will spend $20 + $2 = $<<20+2=22>>22.
Elle and Andrea will spend $1.5 x 4 = $<<1.5*4=6>>6 for the soda.
Elle and Andrea will spend $6 + $10 = $<<6+10=16>>16 for the soda and chicken wings.
Elle, Andrea, and Joe together will spend $16 + $5 = $<<16+5=21>>21.
So, Mary will spend $22 - $21 = $<<22-21=1>>1 more than all of them combined.
#### 1

Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?
Answer: The grill burned 3 * 60 = <<3*60=180>>180 coals.
It takes 20 minutes to burn 15 coals, so the grill ran for 180 / 15 * 20 = <<180/15*20=240>>240 minutes.
#### 240

Question: A bear is preparing to hibernate for the winter and needs to gain 1000 pounds. At the end of summer, the bear feasts on berries and small woodland animals. During autumn, it devours acorns and salmon. It gained a fifth of the weight it needed from berries during summer, and during autumn, it gained twice that amount from acorns. Salmon made up half of the remaining weight it had needed to gain. How many pounds did it gain eating small animals?
Answer: The bear gained 1 / 5 * 1000 = <<1/5*1000=200>>200 pounds from berries.
It gained 2 * 200 = <<2*200=400>>400 pounds from acorns.
It still needed 1000 - 200 - 400 = <<1000-200-400=400>>400 pounds.
Thus, it gained 400 / 2 = <<400/2=200>>200 pounds from salmon.
Therefore, the bear gained 400 - 200 = <<400-200=200>>200 pounds from small animals.
#### 200

Question: Brendan can cut 8 yards of grass per day, he bought a lawnmower and it helped him to cut more yards by Fifty percent per day. How many yards will Brendan be able to cut after a week?
Answer: The additional yard Brendan can cut after buying the lawnmower is 8 x 0.50 = <<8*0.50=4>>4 yards.
So, the total yards he can cut with the lawnmower is 8 + 4 = <<8+4=12>>12.
Therefore, the total number of yards he can cut in a week is 12 x 7 = <<12*7=84>>84 yards.
#### 84"""


def build_prompt(question: str) -> torch.Tensor:
    text = FEW_SHOT_EXAMPLES + f'\n\nQuestion: {question}\nAnswer:'
    messages = [{'role': 'user', 'content': text}]
    formatted = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    ids = tokenizer(formatted)['input_ids']
    return torch.tensor(ids, dtype=torch.long, device='cuda').unsqueeze(0)


def extract_answer(text: str) -> str | None:
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    return m.group(1).replace(',', '').strip() if m else None


LIMIT = 50
prompts = [build_prompt(gsm8k[i]['question']) for i in range(LIMIT)]
questions = [gsm8k[i]['question'] for i in range(LIMIT)]
ref_answers = [gsm8k[i]['answer'] for i in range(LIMIT)]
print(f'Built {len(prompts)} prompts, first length: {prompts[0].shape[1]} tokens')

## 4. 扫参配置

In [ ]:
SWEEP_RATIOS = [round(r, 1) for r in np.arange(0.1, 1.0, 0.1).tolist()]

GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 32
THRESHOLD = 0.9

REWARM_ON_EXPAND = True
FRONT_BLOCK_FALLBACK_ONLY = True

print(f'Sweep ratios: {SWEEP_RATIOS}')
print(f'Samples per ratio: {LIMIT}')
print(f'Total generations: {len(SWEEP_RATIOS) * LIMIT}')

## 5. 跑 sweep + 生成 HTML

每个 ratio 跑完 50 个 sample 后立即生成 HTML 并释放内存。

预计耗时：~7 分钟/ratio × 9 = ~63 分钟（单 GPU）。

In [ ]:
from viz_static import generate_with_collection_expand, process_sample, generate_multi_html
import time

OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, 'viz_sweep')
os.makedirs(OUTPUT_DIR, exist_ok=True)

summary = []  # (ratio, acc, total_nfe, time, html_path, html_size_mb)

for ratio in SWEEP_RATIOS:
    print(f'\n{"="*60}')
    print(f'  mid_trigger_ratio = {ratio:.1f}')
    print(f'{"="*60}')

    all_samples = []
    total_nfe = 0
    correct = 0
    t0 = time.time()

    for i in range(LIMIT):
        prompt = prompts[i]

        x, nfe, history = generate_with_collection_expand(
            model, prompt,
            steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
            temperature=0.0, threshold=THRESHOLD, mask_id=MASK_ID,
            mid_trigger_ratio=ratio,
            rewarm_on_expand=REWARM_ON_EXPAND,
            front_block_fallback_only=FRONT_BLOCK_FALLBACK_ONLY,
        )
        total_nfe += nfe

        gen_text = tokenizer.decode(x[0, prompt.shape[1]:], skip_special_tokens=True)
        for stop in ['Question:', '\n\nQuestion']:
            if stop in gen_text:
                gen_text = gen_text.split(stop)[0]
        gen_ans = extract_answer(gen_text)
        ref_ans = extract_answer(ref_answers[i])
        is_correct = gen_ans is not None and ref_ans is not None and gen_ans == ref_ans
        if is_correct:
            correct += 1

        sample_json = process_sample(
            history, tokenizer,
            question=questions[i],
            gen_answer=f"{gen_ans} {'✓' if is_correct else '✗ (ref: '+str(ref_ans)+')'}",
            ref_answer=ref_ans,
        )
        all_samples.append(sample_json)

        if (i + 1) % 10 == 0 or i == LIMIT - 1:
            elapsed = time.time() - t0
            print(f'  [{i+1}/{LIMIT}] NFE={nfe:3d}  acc={correct}/{i+1}  ({elapsed:.0f}s)')

    elapsed = time.time() - t0
    acc = correct / LIMIT

    html = generate_multi_html(
        all_samples,
        title=f'GSM8K Denoising — ratio={ratio:.1f} (acc={acc:.2%}, NFE={total_nfe})',
    )
    html_path = os.path.join(OUTPUT_DIR, f'viz_ratio_{ratio:.2f}.html')
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html)
    size_mb = os.path.getsize(html_path) / 1024 / 1024

    summary.append((ratio, acc, total_nfe, elapsed, html_path, size_mb))
    print(f'  → acc={acc:.2%}  NFE={total_nfe}  time={elapsed:.0f}s  HTML={size_mb:.1f}MB')

    del all_samples, html
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('All done!')

## 6. 汇总

In [ ]:
import pandas as pd

df = pd.DataFrame(summary, columns=['ratio', 'accuracy', 'total_nfe', 'time_sec', 'html_path', 'html_size_mb'])
display(df[['ratio', 'accuracy', 'total_nfe', 'time_sec', 'html_size_mb']])

print(f'\nHTML files saved to: {OUTPUT_DIR}/')
for _, row in df.iterrows():
    print(f'  ratio={row["ratio"]:.1f}  acc={row["accuracy"]:.2%}  → {os.path.basename(row["html_path"])} ({row["html_size_mb"]:.1f} MB)')
print(f'\n下载 viz_sweep/ 目录到本地，用浏览器打开 HTML 即可浏览 denoising 过程')

## 7. Accuracy & NFE vs ratio（快速对比图）

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df['ratio'], df['accuracy'], 'o-', color='#2196F3', lw=2, ms=6)
ax1.set_xlabel('mid_trigger_ratio'); ax1.set_ylabel('Accuracy')
ax1.set_title('GSM8K Accuracy vs mid_trigger_ratio', fontweight='bold')
ax1.grid(True, alpha=0.3)
for _, row in df.iterrows():
    ax1.annotate(f'{row["accuracy"]:.2%}', (row['ratio'], row['accuracy']),
                 textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

ax2.plot(df['ratio'], df['total_nfe'], 's-', color='#FF5722', lw=2, ms=6)
ax2.set_xlabel('mid_trigger_ratio'); ax2.set_ylabel('Total NFE (50 samples)')
ax2.set_title('NFE vs mid_trigger_ratio', fontweight='bold')
ax2.grid(True, alpha=0.3)
for _, row in df.iterrows():
    ax2.annotate(str(int(row['total_nfe'])), (row['ratio'], row['total_nfe']),
                 textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

fig.suptitle(f'Mid-Trigger-Ratio Sweep (GSM8K, {LIMIT} samples)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sweep_summary.png'), dpi=150, bbox_inches='tight')
plt.show()